# Data Pipeline

End-to-end data preparation notebook for cross-exchange arbitrage research.

This notebook does two stages in sequence:
1. Build / load composite orderbook replay pages from venue persisted files.
2. Scan composite pages for cross-exchange arbitrage candidates.

## Inputs
- Persisted venue files from `persist/pop-data` (`.jsonl` / `.jsonl.gz`).
- Optional existing cache: `persist/analysis-cache/composite_pages_btcusdt.pkl.gz`.

## Outputs
- Composite replay cache: `persist/analysis-cache/composite_pages_btcusdt.pkl.gz`.
- Arbitrage candidate cache: `persist/analysis-cache/cross_exchange_arbitrage_btcusdt2.pkl.gz`.
- Optional CSV export: `persist/analysis-cache/cross_exchange_arbitrage_btcusdt2.csv`.

## Recommended Run Order
1. Run top-to-bottom once after new market data is collected.
2. If cache is enabled, confirm the cache metadata shown by the load cell.
3. Validate read counters before trusting downstream candidate stats.

## Notes
- This notebook is deterministic data prep. Keep simulation logic in `simulation_and_analysis.ipynb`.
- If you want a full rebuild, disable cache usage in settings cells and rerun.



## Part A: Composite Orderbook Build


## Configuration Guide

Main configuration blocks in this notebook:
- **Input discovery**: `INPUTS`, `SYMBOL_FILTER`.
- **Freshness control**: `MAX_BOOK_AGE_MS` (venue book age tolerance in replay).
- **Cache control**: `USE_CACHE_IF_AVAILABLE`, `PAGES_CACHE_PATH`.
- **Candidate filters**: `MIN_SPREAD_ABS`, `MIN_SPREAD_BPS`, `ONLY_CROSSED`, freshness limits.
- **Execution realism assumptions**: venue fees, slippage, latency haircut, fixed cost.

When debugging no-results scenarios, check in this order:
1. Read counters (`READ_STATS`) and accepted row count.
2. Symbol normalization/filter (`SYMBOL_FILTER`).
3. Candidate filter thresholds and `ONLY_CROSSED`.
4. Cache freshness vs current persisted files.



# Composite Order Book

Stage 1 notebook: configure inputs and load persisted venue `book_state` records. Composite aggregation logic will be added afterward.

## Settings

This first stage only does two things:
- define the input and freshness settings
- load and validate persisted `book_state` rows from the venue files

It does not yet build the composite order book.

In [ ]:
from __future__ import annotations

import gzip
import json
from pprint import pprint
import zlib
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

# INPUTS: Directories or files to scan. Supported formats are .jsonl and .jsonl.gz.
INPUTS = [Path("/home/fkonrad97/projects/pop/persist/pop-data")]
# SYMBOL_FILTER: Canonical symbol after normalization. BTCUSDT matches BTC-USDT,
# BTCUSDT, and btcusdt@depth@100ms. Set to None to load every symbol found.
SYMBOL_FILTER = "BTCUSDT"
# MAX_BOOK_AGE_MS: Freshness threshold that will be used later when deciding which
# venue books are still usable for a composite view.
MAX_BOOK_AGE_MS = 2500
# USE_CACHE_IF_AVAILABLE: If True and a cached replay exists, load it instead of
# rereading persisted venue files and rebuilding pages_df.
USE_CACHE_IF_AVAILABLE = True
PAGES_CACHE_PATH = Path("/home/fkonrad97/projects/pop/persist/analysis-cache/composite_pages_btcusdt.pkl.gz")
skip_source_replay = False
pages_df = pd.DataFrame()
books_df = pd.DataFrame()

# READ_STATS: Filled during parsing so it is obvious whether data was actually read.
READ_STATS = {
    "files_seen": 0,
    "book_state_rows": 0,
    "accepted_rows": 0,
    "skipped_non_book_state": 0,
    "skipped_invalid_json": 0,
    "skipped_missing_required_fields": 0,
    "skipped_invalid_timestamp": 0,
    "skipped_missing_top_of_book": 0,
}


def sanitize_pages_df_for_pickle(df: pd.DataFrame) -> pd.DataFrame:
    """Convert pandas extension dtypes into plain pickle-friendly Python/object dtypes."""
    if df.empty:
        return df.copy()
    sanitized = df.copy()
    sanitized.columns = [str(col) for col in sanitized.columns]
    for column in sanitized.columns:
        series = sanitized[column]
        if pd.api.types.is_string_dtype(series.dtype):
            sanitized[column] = series.astype(object)
        elif pd.api.types.is_integer_dtype(series.dtype):
            sanitized[column] = series.astype('int64')
        elif pd.api.types.is_float_dtype(series.dtype):
            sanitized[column] = series.astype('float64')
    return sanitized


In [ ]:
@dataclass(frozen=True)
class Level:
    price: Decimal
    quantity: Decimal
    price_raw: str
    quantity_raw: str


@dataclass(frozen=True)
class BookState:
    venue: str
    symbol: str
    canonical_symbol: str
    ts_book_ns: int
    top_n: int
    bids: list[Level]
    asks: list[Level]
    source_path: Path


def is_supported_input(path: Path) -> bool:
    """Return True for persisted file types the notebook knows how to parse."""
    name = path.name.lower()
    return name.endswith(".jsonl") or name.endswith(".jsonl.gz")


def expand_inputs(inputs: list[Path]) -> list[Path]:
    """Expand input directories into concrete persisted file paths."""
    files: list[Path] = []
    for root in inputs:
        if root.is_dir():
            for candidate in sorted(root.iterdir()):
                if candidate.is_file() and is_supported_input(candidate):
                    files.append(candidate)
        elif root.is_file() and is_supported_input(root):
            files.append(root)
    return files


def open_text(path: Path):
    """Open plain JSONL or gzip-compressed JSONL in text mode."""
    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return path.open("rt", encoding="utf-8")


def normalize_symbol(symbol: str) -> str:
    """Map venue-specific symbols into one comparable canonical symbol."""
    base = symbol.split("@", 1)[0]
    return "".join(ch for ch in base.upper() if ch.isalnum())


def parse_decimal(value: object) -> Decimal | None:
    """Safely convert persisted numeric fields into Decimal values."""
    if value is None:
        return None
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError):
        return None


def parse_levels(levels: object) -> list[Level]:
    """Parse one persisted side of the book and drop malformed or empty levels."""
    parsed: list[Level] = []
    if not isinstance(levels, list):
        return parsed
    for raw in levels:
        if not isinstance(raw, dict):
            continue
        price_raw = str(raw.get("price", "")).strip()
        quantity_raw = str(raw.get("quantity", "")).strip()
        price = parse_decimal(price_raw)
        quantity = parse_decimal(quantity_raw)
        if price is None or quantity is None or quantity <= 0:
            continue
        parsed.append(
            Level(
                price=price,
                quantity=quantity,
                price_raw=price_raw,
                quantity_raw=quantity_raw,
            )
        )
    return parsed


def is_valid_book_state_obj(obj: dict) -> bool:
    """Check that a parsed JSON object has the minimum book_state fields we require."""
    required = ["venue", "symbol", "ts_book_ns", "bids", "asks"]
    return all(field in obj for field in required)


def iter_book_states(paths: list[Path]):
    """Yield validated BookState rows from persisted venue files.

    The reader tolerates live .jsonl.gz files that may currently be mid-write by
    stopping at the last decodable point instead of failing the whole notebook.
    """
    for path in paths:
        READ_STATS["files_seen"] += 1
        try:
            with open_text(path) as handle:
                while True:
                    try:
                        line = handle.readline()
                    except (EOFError, gzip.BadGzipFile, zlib.error, OSError):
                        break
                    if not line:
                        break
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                    except json.JSONDecodeError:
                        READ_STATS["skipped_invalid_json"] += 1
                        continue
                    if obj.get("event_type") != "book_state":
                        READ_STATS["skipped_non_book_state"] += 1
                        continue
                    READ_STATS["book_state_rows"] += 1
                    if not is_valid_book_state_obj(obj):
                        READ_STATS["skipped_missing_required_fields"] += 1
                        continue
                    venue = str(obj.get("venue", "")).strip()
                    symbol = str(obj.get("symbol", "")).strip()
                    ts_book_ns = int(obj.get("ts_book_ns", 0))
                    top_n = int(obj.get("top_n", 0))
                    if not venue or not symbol or ts_book_ns <= 0:
                        READ_STATS["skipped_invalid_timestamp"] += 1
                        continue
                    bids = parse_levels(obj.get("bids"))
                    asks = parse_levels(obj.get("asks"))
                    if not bids or not asks:
                        READ_STATS["skipped_missing_top_of_book"] += 1
                        continue
                    READ_STATS["accepted_rows"] += 1
                    yield BookState(
                        venue=venue,
                        symbol=symbol,
                        canonical_symbol=normalize_symbol(symbol),
                        ts_book_ns=ts_book_ns,
                        top_n=top_n,
                        bids=bids,
                        asks=asks,
                        source_path=path,
                    )
        except (gzip.BadGzipFile, zlib.error, OSError):
            continue


## Load Persisted Venue Books

This cell reads the persisted venue files, validates each `book_state`, applies the optional symbol filter, and shows a compact summary plus a few sample rows.

In [ ]:
skip_source_replay = False
input_paths = expand_inputs(INPUTS)
symbol_filter = normalize_symbol(SYMBOL_FILTER) if SYMBOL_FILTER else None

if USE_CACHE_IF_AVAILABLE and PAGES_CACHE_PATH.exists():
    pages_df = sanitize_pages_df_for_pickle(pd.read_pickle(PAGES_CACHE_PATH, compression="gzip"))
    skip_source_replay = True
    states = []
    books_df = pd.DataFrame()
    read_summary = dict(READ_STATS)
    sample_rows = []
    {
        "cache_loaded": True,
        "cache_path": str(PAGES_CACHE_PATH),
        "pages_rows": int(len(pages_df)),
        "input_paths": [str(path) for path in input_paths],
        "symbol_filter": symbol_filter,
        "state_count": 0,
        "read_summary": read_summary,
        "sample_rows": "Source ingest skipped because cached replay was loaded.",
    }
else:
    states = sorted(iter_book_states(input_paths), key=lambda item: item.ts_book_ns)
    if symbol_filter:
        states = [state for state in states if state.canonical_symbol == symbol_filter]

    read_summary = dict(READ_STATS)
    sample_rows = [
        {
            "venue": state.venue,
            "symbol": state.symbol,
            "canonical_symbol": state.canonical_symbol,
            "ts_book_ns": state.ts_book_ns,
            "top_n": state.top_n,
            "best_bid": state.bids[0].price_raw if state.bids else None,
            "best_ask": state.asks[0].price_raw if state.asks else None,
            "source_path": str(state.source_path),
        }
        for state in states[:10]
    ]

    {
        "cache_loaded": False,
        "input_paths": [str(path) for path in input_paths],
        "symbol_filter": symbol_filter,
        "state_count": len(states),
        "read_summary": read_summary,
        "sample_rows": sample_rows if sample_rows else "No matching book_state rows found.",
    }


## Read Summary

This cell pretty-prints the ingest counters so it is obvious how many files and `book_state` rows were accepted or skipped during parsing.

In [ ]:
print("READ_STATS")
pprint(read_summary, sort_dicts=False)


## Convert Loaded States To DataFrame

This cell flattens the validated `BookState` objects into a pandas DataFrame so the next composite-book steps can group, filter, and transform the venue books more easily.

In [ ]:
rows = [
    {
        "venue": state.venue,
        "symbol": state.symbol,
        "canonical_symbol": state.canonical_symbol,
        "ts_book_ns": state.ts_book_ns,
        "top_n": state.top_n,
        "source_path": str(state.source_path),
        "best_bid_price": str(state.bids[0].price) if state.bids else None,
        "best_bid_qty": str(state.bids[0].quantity) if state.bids else None,
        "best_ask_price": str(state.asks[0].price) if state.asks else None,
        "best_ask_qty": str(state.asks[0].quantity) if state.asks else None,
        "bid_levels": [
            {"price": level.price_raw, "quantity": level.quantity_raw}
            for level in state.bids
        ],
        "ask_levels": [
            {"price": level.price_raw, "quantity": level.quantity_raw}
            for level in state.asks
        ],
    }
    for state in states
]

if skip_source_replay:
    books_df = pd.DataFrame()
    display({
        "skipped": True,
        "reason": "cached replay already loaded",
        "cache_path": str(PAGES_CACHE_PATH),
        "pages_rows": int(len(pages_df)) if not pages_df.empty else 0,
    })
else:
    books_df = pd.DataFrame(rows)

if skip_source_replay:
    pass
elif books_df.empty:
    display("No rows loaded into DataFrame.")
else:
    books_df["ts_book_dt"] = pd.to_datetime(books_df["ts_book_ns"], unit="ns")
    numeric_columns = ["best_bid_price", "best_bid_qty", "best_ask_price", "best_ask_qty"]
    for column in numeric_columns:
        books_df[column] = pd.to_numeric(books_df[column])
    display({
        "row_count": len(books_df),
        "venues": sorted(books_df["venue"].unique()),
        "symbols": sorted(books_df["canonical_symbol"].unique()),
    })
    display(books_df.head(10))


## Build Latest Venue Book Per Exchange

A unified order book should use one checkpoint per venue, not every historical checkpoint. This cell keeps the latest checkpoint per venue and drops venue books that are older than `MAX_BOOK_AGE_MS` relative to the newest venue book.

In [ ]:
if books_df.empty:
    latest_books_df = pd.DataFrame()
    display("No venue books available.")
else:
    latest_ts_book_ns = int(books_df["ts_book_ns"].max())
    latest_books_df = (
        books_df.sort_values(["venue", "ts_book_ns"])
        .groupby("venue", as_index=False)
        .tail(1)
        .copy()
    )
    latest_books_df["age_ms"] = (latest_ts_book_ns - latest_books_df["ts_book_ns"]) / 1_000_000
    latest_books_df = latest_books_df[latest_books_df["age_ms"] <= MAX_BOOK_AGE_MS].copy()
    latest_books_df.sort_values(["venue"], inplace=True)
    latest_books_df.reset_index(drop=True, inplace=True)
    display({
        "latest_ts_book_ns": latest_ts_book_ns,
        "fresh_venue_books": len(latest_books_df),
        "venues": latest_books_df["venue"].tolist(),
    })
    display(latest_books_df[[
        "venue",
        "symbol",
        "canonical_symbol",
        "ts_book_dt",
        "age_ms",
        "best_bid_price",
        "best_ask_price",
        "top_n",
    ]])


## Unified Order Book Table

This cell explodes the latest fresh venue books into one combined table of levels. Bids are sorted from highest to lowest, asks from lowest to highest, and each row keeps the source venue.

In [ ]:
def explode_levels(latest_books_df: pd.DataFrame, side: str) -> pd.DataFrame:
    """Flatten bid_levels or ask_levels from the latest venue books into one table."""
    if latest_books_df.empty:
        return pd.DataFrame(columns=["side", "level_index", "venue", "ts_book_ns", "ts_book_dt", "age_ms", "price", "quantity"])

    level_column = "bid_levels" if side == "bid" else "ask_levels"
    rows: list[dict[str, object]] = []
    for _, book in latest_books_df.iterrows():
        for level_index, level in enumerate(book[level_column], start=1):
            rows.append({
                "side": side,
                "level_index": level_index,
                "venue": book["venue"],
                "ts_book_ns": book["ts_book_ns"],
                "ts_book_dt": book["ts_book_dt"],
                "age_ms": book["age_ms"],
                "price": pd.to_numeric(level["price"]),
                "quantity": pd.to_numeric(level["quantity"]),
            })
    exploded = pd.DataFrame(rows)
    if exploded.empty:
        return exploded
    ascending = [False, True, True] if side == "bid" else [True, True, True]
    exploded.sort_values(["price", "venue", "level_index"], ascending=ascending, inplace=True)
    exploded.reset_index(drop=True, inplace=True)
    return exploded


unified_bids_df = explode_levels(latest_books_df, "bid")
unified_asks_df = explode_levels(latest_books_df, "ask")

display("Unified bids")
display(unified_bids_df.head(20))
display("Unified asks")
display(unified_asks_df.head(20))


## Replay Composite Book Pages

The cells above show only the terminal latest state. This section replays the venue checkpoints over time and creates one page per unique `ts_book_ns`. Each page represents the unified order book state available at that moment, using the latest known checkpoint from each venue and dropping books older than `MAX_BOOK_AGE_MS`.

In [ ]:
def build_page(ts_book_ns: int, latest_by_venue: dict[str, dict[str, object]]) -> dict[str, object]:
    """Build one composite-book page for a single event timestamp."""
    page_ts = pd.to_datetime(ts_book_ns, unit="ns")
    if not latest_by_venue:
        return {
            "ts_book_ns": ts_book_ns,
            "ts_book_dt": page_ts,
            "venue_count": 0,
            "venues": [],
            "best_bid": None,
            "best_ask": None,
            "spread": None,
            "venue_books": [],
            "unified_bids": [],
            "unified_asks": [],
        }

    venue_books: list[dict[str, object]] = []
    for book in latest_by_venue.values():
        age_ms = (ts_book_ns - int(book["ts_book_ns"])) / 1_000_000
        if age_ms > MAX_BOOK_AGE_MS:
            continue
        venue_books.append({
            "venue": book["venue"],
            "symbol": book["symbol"],
            "ts_book_ns": int(book["ts_book_ns"]),
            "ts_book_dt": book["ts_book_dt"],
            "age_ms": age_ms,
            "best_bid_price": float(book["best_bid_price"]),
            "best_ask_price": float(book["best_ask_price"]),
            "top_n": int(book["top_n"]),
            "bid_levels": book["bid_levels"],
            "ask_levels": book["ask_levels"],
        })

    if not venue_books:
        return {
            "ts_book_ns": ts_book_ns,
            "ts_book_dt": page_ts,
            "venue_count": 0,
            "venues": [],
            "best_bid": None,
            "best_ask": None,
            "spread": None,
            "venue_books": [],
            "unified_bids": [],
            "unified_asks": [],
        }

    venue_books.sort(key=lambda book: str(book["venue"]))
    unified_bids: list[dict[str, object]] = []
    unified_asks: list[dict[str, object]] = []

    for book in venue_books:
        for level_index, level in enumerate(book["bid_levels"], start=1):
            unified_bids.append({
                "side": "bid",
                "level_index": level_index,
                "venue": book["venue"],
                "ts_book_ns": book["ts_book_ns"],
                "ts_book_dt": book["ts_book_dt"],
                "age_ms": book["age_ms"],
                "price": float(level["price"]),
                "quantity": float(level["quantity"]),
            })
        for level_index, level in enumerate(book["ask_levels"], start=1):
            unified_asks.append({
                "side": "ask",
                "level_index": level_index,
                "venue": book["venue"],
                "ts_book_ns": book["ts_book_ns"],
                "ts_book_dt": book["ts_book_dt"],
                "age_ms": book["age_ms"],
                "price": float(level["price"]),
                "quantity": float(level["quantity"]),
            })

    unified_bids.sort(key=lambda level: (-level["price"], str(level["venue"]), int(level["level_index"])))
    unified_asks.sort(key=lambda level: (level["price"], str(level["venue"]), int(level["level_index"])))
    best_bid = unified_bids[0]["price"] if unified_bids else None
    best_ask = unified_asks[0]["price"] if unified_asks else None
    spread = best_bid - best_ask if best_bid is not None and best_ask is not None else None

    page_venue_books = [
        {
            "venue": book["venue"],
            "symbol": book["symbol"],
            "ts_book_ns": book["ts_book_ns"],
            "ts_book_dt": book["ts_book_dt"],
            "age_ms": book["age_ms"],
            "best_bid_price": book["best_bid_price"],
            "best_ask_price": book["best_ask_price"],
            "top_n": book["top_n"],
        }
        for book in venue_books
    ]

    return {
        "ts_book_ns": ts_book_ns,
        "ts_book_dt": page_ts,
        "venue_count": len(page_venue_books),
        "venues": [book["venue"] for book in page_venue_books],
        "best_bid": best_bid,
        "best_ask": best_ask,
        "spread": spread,
        "venue_books": page_venue_books,
        "unified_bids": unified_bids,
        "unified_asks": unified_asks,
    }


pages: list[dict[str, object]] = []
if USE_CACHE_IF_AVAILABLE and PAGES_CACHE_PATH.exists():
    if pages_df.empty:
        pages_df = sanitize_pages_df_for_pickle(pd.read_pickle(PAGES_CACHE_PATH, compression="gzip"))
    skip_source_replay = True
    display({
        "cache_loaded": True,
        "cache_path": str(PAGES_CACHE_PATH),
        "page_count": len(pages_df),
        "first_ts_book_ns": int(pages_df.iloc[0]["ts_book_ns"]) if not pages_df.empty else None,
        "last_ts_book_ns": int(pages_df.iloc[-1]["ts_book_ns"]) if not pages_df.empty else None,
    })
    display(pages_df[["ts_book_ns", "ts_book_dt", "venue_count", "venues", "best_bid", "best_ask", "spread"]].head(20))
elif books_df.empty:
    pages_df = pd.DataFrame()
    display("No pages available because books_df is empty.")
else:
    replay_df = books_df.sort_values(["ts_book_ns", "venue"]).reset_index(drop=True)
    latest_by_venue: dict[str, dict[str, object]] = {}
    for ts_book_ns, group in replay_df.groupby("ts_book_ns", sort=True):
        for row in group.to_dict("records"):
            latest_by_venue[row["venue"]] = row
        pages.append(build_page(int(ts_book_ns), latest_by_venue))

    pages_df = pd.DataFrame(pages)
    if not PAGES_CACHE_PATH.exists() and not pages_df.empty:
        PAGES_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        sanitize_pages_df_for_pickle(pages_df).to_pickle(PAGES_CACHE_PATH, compression="gzip")
    display({
        "page_count": len(pages_df),
        "cache_saved": PAGES_CACHE_PATH.exists(),
        "cache_path": str(PAGES_CACHE_PATH),
        "first_ts_book_ns": int(pages_df.iloc[0]["ts_book_ns"]) if not pages_df.empty else None,
        "last_ts_book_ns": int(pages_df.iloc[-1]["ts_book_ns"]) if not pages_df.empty else None,
    })
    display(pages_df[["ts_book_ns", "ts_book_dt", "venue_count", "venues", "best_bid", "best_ask", "spread"]].head(20))


In [ ]:
from pathlib import Path

cache_path = PAGES_CACHE_PATH

if not cache_path.exists():
    print(f"cache file not found: {cache_path}")
else:
    pages_df = sanitize_pages_df_for_pickle(pd.read_pickle(cache_path, compression="gzip"))
    skip_source_replay = True
    print(f"loaded {len(pages_df)} rows from {cache_path}")
    display(pages_df.head(5))


## Inspect One Page

Use this cell to inspect one page from `pages_df`. `PAGE_INDEX = 0` is the earliest unified state, while `PAGE_INDEX = len(pages_df) - 1` is the latest one.

In [ ]:
PAGE_INDEX = len(pages_df) - 1 if 'pages_df' in globals() and not pages_df.empty else 0

if 'pages_df' not in globals() or pages_df.empty:
    display("No page data available.")
else:
    page = pages_df.iloc[PAGE_INDEX]
    display({
        "page_index": PAGE_INDEX,
        "ts_book_ns": int(page["ts_book_ns"]),
        "ts_book_dt": page["ts_book_dt"],
        "venue_count": int(page["venue_count"]),
        "venues": page["venues"],
        "best_bid": page["best_bid"],
        "best_ask": page["best_ask"],
        "spread": page["spread"],
    })
    display(pd.DataFrame(page["venue_books"]))
    display("Unified bids")
    display(pd.DataFrame(page["unified_bids"]).head(20))
    display("Unified asks")
    display(pd.DataFrame(page["unified_asks"]).head(20))


## Replay Controls

This is a notebook-friendly replay flow without frontend widgets. Set the page and depth values, rerun the render cell, and use the helper functions to move through the saved pages.

In [ ]:
PAGE_INDEX = len(pages_df) - 1 if "pages_df" in globals() and not pages_df.empty else 0
DEPTH = 10

def clamp_page_index(page_index: int) -> int:
    """Clamp the requested page index into the valid pages_df range."""
    if "pages_df" not in globals() or pages_df.empty:
        return 0
    return max(0, min(int(page_index), len(pages_df) - 1))


def show_page(page_index: int, depth: int = 10):
    """Render one replay page with venue books and unified bids/asks."""
    if "pages_df" not in globals() or pages_df.empty:
        display("No page data available for replay.")
        return
    page_index = clamp_page_index(page_index)
    page = pages_df.iloc[page_index]
    display({
        "page_index": int(page_index),
        "ts_book_ns": int(page["ts_book_ns"]),
        "ts_book_dt": page["ts_book_dt"],
        "venue_count": int(page["venue_count"]),
        "venues": page["venues"],
        "best_bid": page["best_bid"],
        "best_ask": page["best_ask"],
        "spread": page["spread"],
    })
    display(pd.DataFrame(page["venue_books"]))
    display("Unified bids")
    display(pd.DataFrame(page["unified_bids"]).head(depth))
    display("Unified asks")
    display(pd.DataFrame(page["unified_asks"]).head(depth))


def show_next(depth: int = 10):
    """Render the next replay page."""
    global PAGE_INDEX
    PAGE_INDEX = clamp_page_index(PAGE_INDEX + 1)
    show_page(PAGE_INDEX, depth)


def show_prev(depth: int = 10):
    """Render the previous replay page."""
    global PAGE_INDEX
    PAGE_INDEX = clamp_page_index(PAGE_INDEX - 1)
    show_page(PAGE_INDEX, depth)


def jump_to_latest(depth: int = 10):
    """Render the latest replay page."""
    global PAGE_INDEX
    PAGE_INDEX = clamp_page_index(len(pages_df) - 1 if "pages_df" in globals() and not pages_df.empty else 0)
    show_page(PAGE_INDEX, depth)


In [ ]:
show_page(PAGE_INDEX, DEPTH)


## Part B: Cross-Exchange Arbitrage Candidate Build


# Cross-Exchange Arbitrage

Load the cached unified order book replay and check each page for cross-exchange arbitrage candidates.

## Scope

This notebook works only from the cached unified replay (`pages_df`).

It:
- loads the cached unified order book pages
- converts them into a pandas DataFrame
- inspects each page for a crossed market between different venues
- reports candidate opportunities with buy venue, sell venue, spread, and freshness

It does not yet model fees, slippage, inventory, or depth-walking execution.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

# CACHE_PATH: Cached unified replay generated by composite_orderbook.ipynb.
CACHE_PATH = Path("/home/fkonrad97/projects/pop/persist/analysis-cache/composite_pages_btcusdt.pkl.gz")
# MIN_SPREAD_ABS: Minimum absolute spread required to keep a candidate.
MIN_SPREAD_ABS = 0.0
# MIN_SPREAD_BPS: Minimum spread in basis points required to keep a candidate.
MIN_SPREAD_BPS = 0.0
# ONLY_CROSSED: If True, keep only pages where sell bid exceeds buy ask.
ONLY_CROSSED = True
# VENUE_TAKER_FEE_BPS: Assumed taker fee per venue in basis points.
VENUE_TAKER_FEE_BPS = {
    "binance": 10.0,
    "okx": 10.0,
    "bybit": 10.0,
    "kucoin": 10.0,
}
# ASSUMED_SLIPPAGE_BPS_PER_SIDE: Extra per-side slippage assumption.
ASSUMED_SLIPPAGE_BPS_PER_SIDE = 0.0
# LATENCY_HAIRCUT_BPS: Additional safety haircut in basis points.
LATENCY_HAIRCUT_BPS = 0.0
# FIXED_COST_IN_QUOTE: Flat extra round-trip cost in quote currency.
FIXED_COST_IN_QUOTE = 0.0
# MIN_EXECUTABLE_QTY: Minimum top-level size required on both sides.
MIN_EXECUTABLE_QTY = 0.0
# REQUIRE_FRESH_VENUE_AGES_MS: Optional per-side freshness limit. Set to None to disable.
REQUIRE_FRESH_VENUE_AGES_MS = None
# DEDUPLICATE_CONSECUTIVE: If True, collapse repeated sequential candidates that keep
# the same effective opportunity state.
DEDUPLICATE_CONSECUTIVE = True
# DEDUP_PRICE_EPSILON: Treat tiny price moves within this threshold as unchanged.
DEDUP_PRICE_EPSILON = 0.0
# DEDUP_QTY_EPSILON: Treat tiny quantity moves within this threshold as unchanged.
DEDUP_QTY_EPSILON = 0.0
# LIMIT_ROWS: Number of opportunity rows to preview.
LIMIT_ROWS = 20


In [ ]:
def sanitize_pages_df(df: pd.DataFrame) -> pd.DataFrame:
    """Convert extension dtypes into plain pandas-friendly dtypes after pickle load."""
    if df.empty:
        return df.copy()
    sanitized = df.copy()
    sanitized.columns = [str(col) for col in sanitized.columns]
    for column in sanitized.columns:
        series = sanitized[column]
        if pd.api.types.is_string_dtype(series.dtype):
            sanitized[column] = series.astype(object)
        elif pd.api.types.is_integer_dtype(series.dtype):
            sanitized[column] = series.astype("int64")
        elif pd.api.types.is_float_dtype(series.dtype):
            sanitized[column] = series.astype("float64")
    return sanitized


def load_pages_df(cache_path: Path) -> pd.DataFrame:
    """Load the cached unified replay into a DataFrame."""
    if not cache_path.exists():
        raise FileNotFoundError(f"Cache file not found: {cache_path}")
    return sanitize_pages_df(pd.read_pickle(cache_path, compression="gzip"))


def page_levels_df(page: pd.Series, side_key: str) -> pd.DataFrame:
    """Convert one page's nested unified bid/ask payload into a DataFrame."""
    records = page.get(side_key, [])
    if not records:
        return pd.DataFrame()
    levels_df = pd.DataFrame(records).copy()
    if levels_df.empty:
        return levels_df
    levels_df["price"] = pd.to_numeric(levels_df["price"])
    levels_df["quantity"] = pd.to_numeric(levels_df["quantity"])
    levels_df["age_ms"] = pd.to_numeric(levels_df["age_ms"])
    return levels_df


def best_cross_exchange_pair(page: pd.Series) -> dict[str, object] | None:
    """Find the best buy/sell pair on one page across different venues."""
    bids_df = page_levels_df(page, "unified_bids")
    asks_df = page_levels_df(page, "unified_asks")
    if bids_df.empty or asks_df.empty:
        return None

    best_bid = bids_df.iloc[0]
    best_ask = asks_df.iloc[0]

    if best_bid["venue"] != best_ask["venue"]:
        chosen_bid = best_bid
        chosen_ask = best_ask
    else:
        chosen_bid = None
        chosen_ask = None
        for _, ask in asks_df.iterrows():
            for _, bid in bids_df.iterrows():
                if ask["venue"] == bid["venue"]:
                    continue
                chosen_ask = ask
                chosen_bid = bid
                break
            if chosen_bid is not None:
                break

        if chosen_bid is None or chosen_ask is None:
            return None

    spread_abs = float(chosen_bid["price"] - chosen_ask["price"])
    mid = float((chosen_bid["price"] + chosen_ask["price"]) / 2.0)
    spread_bps = (spread_abs / mid) * 10000.0 if mid > 0 else None

    return {
        "ts_book_ns": int(page["ts_book_ns"]),
        "ts_book_dt": page["ts_book_dt"],
        "venue_count": int(page["venue_count"]),
        "buy_venue": chosen_ask["venue"],
        "buy_price": float(chosen_ask["price"]),
        "buy_qty": float(chosen_ask["quantity"]),
        "buy_age_ms": float(chosen_ask["age_ms"]),
        "sell_venue": chosen_bid["venue"],
        "sell_price": float(chosen_bid["price"]),
        "sell_qty": float(chosen_bid["quantity"]),
        "sell_age_ms": float(chosen_bid["age_ms"]),
        "spread_abs": spread_abs,
        "spread_bps": spread_bps,
        "crossed": spread_abs > 0,
    }


def candidate_signature(candidate: dict[str, object]) -> tuple:
    """Build a dedup signature for one candidate opportunity."""
    return (
        candidate["buy_venue"],
        candidate["sell_venue"],
        round(float(candidate["buy_price"]) / max(DEDUP_PRICE_EPSILON, 1e-12)) if DEDUP_PRICE_EPSILON > 0 else float(candidate["buy_price"]),
        round(float(candidate["sell_price"]) / max(DEDUP_PRICE_EPSILON, 1e-12)) if DEDUP_PRICE_EPSILON > 0 else float(candidate["sell_price"]),
        round(float(candidate["buy_qty"]) / max(DEDUP_QTY_EPSILON, 1e-12)) if DEDUP_QTY_EPSILON > 0 else float(candidate["buy_qty"]),
        round(float(candidate["sell_qty"]) / max(DEDUP_QTY_EPSILON, 1e-12)) if DEDUP_QTY_EPSILON > 0 else float(candidate["sell_qty"]),
    )


def deduplicate_consecutive_candidates(candidates: list[dict[str, object]]) -> list[dict[str, object]]:
    """Collapse repeated sequential candidates that keep the same effective state."""
    if not candidates:
        return candidates
    deduped = [candidates[0]]
    last_signature = candidate_signature(candidates[0])
    for candidate in candidates[1:]:
        signature = candidate_signature(candidate)
        if signature == last_signature:
            continue
        deduped.append(candidate)
        last_signature = signature
    return deduped


## Load Cached Unified Replay

This cell loads `pages_df` from the cached `.pkl.gz` file and shows the basic shape.

In [ ]:
pages_df = load_pages_df(CACHE_PATH)
pages_df["ts_book_dt"] = pd.to_datetime(pages_df["ts_book_dt"])

{
    "cache_path": str(CACHE_PATH),
    "page_count": int(len(pages_df)),
    "first_ts_book_ns": int(pages_df.iloc[0]["ts_book_ns"]) if not pages_df.empty else None,
    "last_ts_book_ns": int(pages_df.iloc[-1]["ts_book_ns"]) if not pages_df.empty else None,
}


## Scan For Arbitrage Candidates

This cell checks each cached page for the best buy/sell venue pair across different exchanges and builds an opportunities DataFrame.

In [ ]:
candidate_rows = []
for _, page in pages_df.iterrows():
    candidate = best_cross_exchange_pair(page)
    if candidate is None:
        continue
    if ONLY_CROSSED and not candidate["crossed"]:
        continue
    if candidate["spread_abs"] < MIN_SPREAD_ABS:
        continue
    if candidate["spread_bps"] is not None and candidate["spread_bps"] < MIN_SPREAD_BPS:
        continue
    candidate_rows.append(candidate)

raw_candidate_count = len(candidate_rows)
if DEDUPLICATE_CONSECUTIVE:
    candidate_rows = deduplicate_consecutive_candidates(candidate_rows)

arb_df = pd.DataFrame(candidate_rows)
if not arb_df.empty:
    arb_df.sort_values(["ts_book_ns"], inplace=True)
    arb_df.reset_index(drop=True, inplace=True)

{
    "raw_candidate_count": int(raw_candidate_count),
    "deduplicated_candidate_count": int(len(arb_df)),
    "deduplicate_consecutive": DEDUPLICATE_CONSECUTIVE,
    "crossed_only": ONLY_CROSSED,
    "min_spread_abs": MIN_SPREAD_ABS,
    "min_spread_bps": MIN_SPREAD_BPS,
}


## Preview Candidates

Inspect the first candidate rows to verify the buy/sell venue pairing and the observed spread.

In [ ]:
arb_df.head(LIMIT_ROWS) if not arb_df.empty else "No arbitrage candidates found."

## Candidate Summary

This groups the detected opportunities by venue pair so you can see which cross-exchange route appears most often.

In [ ]:
if arb_df.empty:
    "No arbitrage candidates found."
else:
    arb_df.groupby(["buy_venue", "sell_venue"]).agg(
        candidate_count=("ts_book_ns", "count"),
        max_spread_abs=("spread_abs", "max"),
        max_spread_bps=("spread_bps", "max"),
        latest_ts=("ts_book_dt", "max"),
    ).sort_values(["candidate_count", "max_spread_abs"], ascending=[False, False])


In [ ]:
from pathlib import Path

cache_dir = Path("/home/fkonrad97/projects/pop/persist/analysis-cache")
cache_dir.mkdir(parents=True, exist_ok=True)

arb_cache_path = cache_dir / "cross_exchange_arbitrage_btcusdt2.pkl.gz"
arb_df.to_pickle(arb_cache_path, compression="gzip")

print(f"saved to {arb_cache_path}")

In [ ]:
from pathlib import Path

if cache_dir is None:
    print("cache_dir is not set, skipping load")
else:
    arb_cache_path = Path("/home/fkonrad97/projects/pop/persist/analysis-cache/cross_exchange_arbitrage_btcusdt2.pkl.gz")
    
    if not arb_cache_path.exists():
        print(f"cache file not found: {arb_cache_path}")
    else:
        compression = "gzip" if arb_cache_path.suffix == ".gz" else None
        arb_df = pd.read_pickle(arb_cache_path, compression=compression)
        print(f"loaded {len(arb_df)} rows from {arb_cache_path}")
        display(arb_df)

In [ ]:
arb_df.to_csv("/home/fkonrad97/projects/pop/persist/analysis-cache/cross_exchange_arbitrage_btcusdt2.csv", index=False)